# preprocess_pattern_v3.ipynb
**Runs from:** `backend/data generation/`

## Leakage-free pattern features — 30 separate batch files

All target-derived features use `.shift(1)` to exclude current timestep.
Each raw batch has all 168 hours for its edges, so per-edge rolling/lag works correctly.
Output: 30 separate parquet files (~55 MB each). **Never** combined into one file.

In [1]:
import os, glob, pickle, time, warnings, gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

RAW_TS_DIR  = 'data/timeseries'
STATIC_PATH = 'data/static/edges_static.parquet'
PATTERN_DIR = 'data/pattern_features'
SCALER_DIR  = 'data/scalers'
os.makedirs(PATTERN_DIR, exist_ok=True)
os.makedirs(SCALER_DIR, exist_ok=True)

DROP_COLS = ['speed_ratio','current_speed','current_travel_time','delay_seconds','confidence']
TRAIN_END = 107; VAL_END = 143; TTR_CLIP = 5.0

df_static = pd.read_parquet(STATIC_PATH)
if 'free_flow_travel_time' not in df_static.columns:
    df_static['free_flow_travel_time'] = (
        df_static['road_length']/(df_static['free_flow_speed']*1000/3600)).clip(lower=0.5).astype(np.float32)
if 'signals_per_km' not in df_static.columns:
    df_static['signals_per_km'] = (df_static['traffic_signal_count'] /
        (df_static['road_length']/1000).clip(lower=0.01)).astype(np.float32)
df_static['oneway'] = df_static['oneway'].astype(np.float32)
STATIC_COLS = ['road_type_enc','num_lanes','oneway','road_length','susceptibility',
    'traffic_signal_count','intersection_count','free_flow_travel_time',
    'corridor_id','zone_id','lat','lon','signals_per_km']
static_lkp = df_static.set_index('edge_id')[STATIC_COLS].to_dict('index')

all_eids = sorted(df_static['edge_id'].unique())
edge_to_idx = {e:i for i,e in enumerate(all_eids)}
N_EDGES = len(all_eids)

timestamps = pd.date_range(start='2024-07-01', periods=168, freq='h')
ts_to_hour = {ts: i for i, ts in enumerate(timestamps)}
batch_files = sorted(glob.glob(f'{RAW_TS_DIR}/batch_*.parquet'))
inc_map = {1:'inc_type_accident',4:'inc_type_rain',6:'inc_type_jam',
           8:'inc_type_road_closed',9:'inc_type_road_works',11:'inc_type_flooding'}
print(f'Edges: {N_EDGES:,}, Batches: {len(batch_files)}')

Edges: 145,265, Batches: 30


In [2]:
# ── Feature group definitions ─────────────────────────────────────────────────
PATTERN_CONTINUOUS = [
    'travel_time_ratio_lag24','travel_time_ratio_lag48',
    'congestion_level_lag24','congestion_level_lag48',
    'travel_time_ratio_roll6_mean','travel_time_ratio_roll6_std',
    'travel_time_ratio_roll6_max','travel_time_ratio_roll6_min',
    'travel_time_ratio_roll12_mean','travel_time_ratio_roll12_std',
    'congestion_level_roll6_mean','congestion_level_roll6_std',
    'congestion_level_roll6_max','congestion_level_roll6_min',
    'congestion_level_roll12_mean','congestion_level_roll12_std',
    'travel_time_ratio_delta1','travel_time_ratio_delta3',
    'congestion_level_delta1','congestion_level_delta3',
    'hourly_rainfall_mm','rain_log1p','rain_roll3_max','rain_roll6_mean','rain_log1p_roll6',
    'incidents_nearby','incident_roll6_sum','incidents_nearby_roll6_mean',
    'rain_x_suscept','cong_prev_x_lanes','ttr_prev_x_roadtype','rain_x_peak',
    'road_length','susceptibility','signals_per_km','lat','lon',
]
PATTERN_ORDINAL = ['day_of_week','hour_of_day','road_type_enc','num_lanes','incident_severity']
PATTERN_BINARY = [
    'road_closure','incident','monsoon_active','local_train_disruption',
    'is_public_holiday','school_holiday','is_weekend','is_peak_hour','oneway',
    'inc_type_accident','inc_type_rain','inc_type_jam',
    'inc_type_road_closed','inc_type_road_works','inc_type_flooding',
    'traffic_signal_count','intersection_count',
]
PATTERN_CYCLIC = ['target_hour_sin','target_hour_cos','target_dow_sin','target_dow_cos']
PATTERN_FEATURES = PATTERN_CONTINUOUS + PATTERN_ORDINAL + PATTERN_BINARY + PATTERN_CYCLIC
TARGET_TTR = 'travel_time_ratio'; TARGET_CONG = 'congestion_level'

print(f'Features: {len(PATTERN_FEATURES)} (cont={len(PATTERN_CONTINUOUS)}, '
      f'ord={len(PATTERN_ORDINAL)}, bin={len(PATTERN_BINARY)}, cyc={len(PATTERN_CYCLIC)})')

Features: 63 (cont=37, ord=5, bin=17, cyc=4)


In [3]:
# ── Batch engineer function (leakage-free) ────────────────────────────────────
def engineer_batch(bf):
    df = pd.read_parquet(bf)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
    df['travel_time_ratio'] = df['travel_time_ratio'].clip(upper=TTR_CLIP)
    df['hour_of_day'] = df['timestamp'].dt.hour.astype(np.int8)
    df['day_of_week'] = df['timestamp'].dt.dayofweek.astype(np.int8)
    df['hour_index'] = df['timestamp'].map(ts_to_hour).astype(np.int16)
    for col in STATIC_COLS:
        df[col] = df['edge_id'].map(lambda e,c=col: static_lkp.get(e,{}).get(c,0)).astype(np.float32)
    df['edge_idx'] = df['edge_id'].map(edge_to_idx).astype(np.int32)
    df = df.sort_values(['edge_id','hour_index']).reset_index(drop=True)
    grp = df.groupby('edge_id')

    # SHIFTED targets (leakage-free: value at t-1, not t)
    ttr_s = grp['travel_time_ratio'].shift(1)
    cong_s = grp['congestion_level'].shift(1)

    for col in ['travel_time_ratio','congestion_level']:
        df[f'{col}_lag24'] = grp[col].shift(24)
        df[f'{col}_lag48'] = grp[col].shift(48)
    for pfx, s in [('travel_time_ratio',ttr_s),('congestion_level',cong_s)]:
        sg = s.groupby(df['edge_id'])
        df[f'{pfx}_roll6_mean'] = sg.transform(lambda x: x.rolling(6,min_periods=1).mean())
        df[f'{pfx}_roll6_std']  = sg.transform(lambda x: x.rolling(6,min_periods=1).std().fillna(0))
        df[f'{pfx}_roll6_max']  = sg.transform(lambda x: x.rolling(6,min_periods=1).max())
        df[f'{pfx}_roll6_min']  = sg.transform(lambda x: x.rolling(6,min_periods=1).min())
        df[f'{pfx}_roll12_mean']= sg.transform(lambda x: x.rolling(12,min_periods=1).mean())
        df[f'{pfx}_roll12_std'] = sg.transform(lambda x: x.rolling(12,min_periods=1).std().fillna(0))
    for pfx, s in [('travel_time_ratio',ttr_s),('congestion_level',cong_s)]:
        df[f'{pfx}_delta1'] = s.groupby(df['edge_id']).transform(lambda x: x.diff(1)).fillna(0)
        df[f'{pfx}_delta3'] = s.groupby(df['edge_id']).transform(lambda x: x.diff(3)).fillna(0)

    df['rain_log1p'] = np.log1p(df['hourly_rainfall_mm']).astype(np.float32)
    df['is_weekend'] = (df['day_of_week']>=5).astype(np.int8)
    h = df['hour_of_day']
    df['is_peak_hour'] = (((h>=7)&(h<=10))|((h>=17)&(h<=20))).astype(np.int8)
    for code, col in inc_map.items():
        df[col] = (df['incident_type']==code).astype(np.int8)
    df['rain_roll3_max']  = grp['hourly_rainfall_mm'].transform(lambda x: x.rolling(3,min_periods=1).max())
    df['rain_roll6_mean'] = grp['hourly_rainfall_mm'].transform(lambda x: x.rolling(6,min_periods=1).mean())
    df['rain_log1p_roll6']= df.groupby('edge_id')['rain_log1p'].transform(lambda x: x.rolling(6,min_periods=1).mean())
    df['incident_roll6_sum'] = grp['incident'].transform(lambda x: x.rolling(6,min_periods=1).sum())
    df['incidents_nearby_roll6_mean'] = grp['incidents_nearby'].transform(lambda x: x.rolling(6,min_periods=1).mean())
    df['cong_prev_x_lanes']   = cong_s.fillna(0) * df['num_lanes']
    df['ttr_prev_x_roadtype'] = ttr_s.fillna(0)  * df['road_type_enc']
    df['rain_x_suscept'] = df['hourly_rainfall_mm'] * df['susceptibility']
    df['rain_x_peak']    = df['hourly_rainfall_mm'] * df['is_peak_hour']
    df['target_hour_sin'] = np.sin(2*np.pi*df['hour_of_day']/24).astype(np.float32)
    df['target_hour_cos'] = np.cos(2*np.pi*df['hour_of_day']/24).astype(np.float32)
    df['target_dow_sin']  = np.sin(2*np.pi*df['day_of_week']/7).astype(np.float32)
    df['target_dow_cos']  = np.cos(2*np.pi*df['day_of_week']/7).astype(np.float32)

    df[PATTERN_FEATURES] = df[PATTERN_FEATURES].fillna(0).astype(np.float32)
    df['split'] = 'train'
    df.loc[df['hour_index']>TRAIN_END, 'split'] = 'val'
    df.loc[df['hour_index']>VAL_END,   'split'] = 'test'
    df['static_free_flow_travel_time'] = df['free_flow_travel_time']

    # NO duplicate columns: day_of_week/hour_of_day already in PATTERN_FEATURES
    out_cols = (['edge_id','edge_idx','corridor_id','zone_id','hour_index','split',
                 TARGET_TTR, TARGET_CONG, 'static_free_flow_travel_time'] + PATTERN_FEATURES)
    return df[out_cols]

test = engineer_batch(batch_files[0])
print(f'Batch 0: {test.shape}, {test.memory_usage(deep=True).sum()/1e6:.0f} MB')
# Check no duplicate columns
assert len(test.columns) == len(set(test.columns)), f'DUPLICATE COLUMNS: {[c for c in test.columns if list(test.columns).count(c)>1]}'
del test; gc.collect()
print('No duplicate columns ✓')

Batch 0: (840000, 72), 269 MB
No duplicate columns ✓


In [4]:
# ── PASS 1: Fit scalers ───────────────────────────────────────────────────────
print('Pass 1: Fitting scalers...')
t0 = time.perf_counter()
acc_c, acc_t, acc_g = [], [], []
for i, bf in enumerate(batch_files):
    df = engineer_batch(bf)
    tr = df[df['split']=='train']
    if len(tr):
        acc_c.append(tr[PATTERN_CONTINUOUS].values.astype(np.float32))
        acc_t.append(tr[[TARGET_TTR]].values.astype(np.float32))
        acc_g.append(tr[[TARGET_CONG]].values.astype(np.float32))
    del df, tr; gc.collect()
    if (i+1)%10==0: print(f'  {i+1}/{len(batch_files)}')

scaler_pattern_cont = StandardScaler().fit(np.concatenate(acc_c))
scaler_pattern_ttr  = StandardScaler().fit(np.concatenate(acc_t))
scaler_pattern_cong = StandardScaler().fit(np.concatenate(acc_g))
del acc_c, acc_t, acc_g; gc.collect()
print(f'Done in {time.perf_counter()-t0:.1f}s')
print(f'  TTR:  mean={scaler_pattern_ttr.mean_[0]:.4f}, std={scaler_pattern_ttr.scale_[0]:.4f}')
print(f'  Cong: mean={scaler_pattern_cong.mean_[0]:.4f}, std={scaler_pattern_cong.scale_[0]:.4f}')

Pass 1: Fitting scalers...
  10/30
  20/30


MemoryError: Unable to allocate 6.41 MiB for an array with shape (840000,) and data type float64

In [ ]:
# ── PASS 2: Transform → save 30 SEPARATE batch files ─────────────────────────
print('\nPass 2: Transforming and saving...')
t0 = time.perf_counter()
for i, bf in enumerate(batch_files):
    df = engineer_batch(bf)
    df[PATTERN_CONTINUOUS] = scaler_pattern_cont.transform(
        df[PATTERN_CONTINUOUS].values.astype(np.float32)).astype(np.float32)
    df[TARGET_TTR] = scaler_pattern_ttr.transform(
        df[[TARGET_TTR]].values.astype(np.float32)).ravel().astype(np.float32)
    df[TARGET_CONG] = scaler_pattern_cong.transform(
        df[[TARGET_CONG]].values.astype(np.float32)).ravel().astype(np.float32)
    df.to_parquet(f'{PATTERN_DIR}/batch_{i:04d}.parquet', index=False, compression='snappy')
    del df; gc.collect()
    if (i+1)%10==0: print(f'  {i+1}/{len(batch_files)}')

for n,o in [('scaler_pattern_cont',scaler_pattern_cont),
            ('scaler_pattern_ttr',scaler_pattern_ttr),('scaler_pattern_cong',scaler_pattern_cong)]:
    with open(f'{SCALER_DIR}/{n}.pkl','wb') as f: pickle.dump(o,f)
meta = {'pattern_features':PATTERN_FEATURES,'pattern_continuous':PATTERN_CONTINUOUS,
    'pattern_ordinal':PATTERN_ORDINAL,'pattern_binary':PATTERN_BINARY,
    'pattern_cyclic':PATTERN_CYCLIC,'n_features':len(PATTERN_FEATURES),
    'n_continuous':len(PATTERN_CONTINUOUS),'n_edges':N_EDGES,'edge_to_idx':edge_to_idx,
    'n_batches':len(batch_files)}
with open(f'{PATTERN_DIR}/pattern_meta.pkl','wb') as f: pickle.dump(meta,f)

print(f'\nSaved {len(batch_files)} separate files → {PATTERN_DIR}/batch_XXXX.parquet')
print(f'Saved scalers + pattern_meta.pkl')
print(f'Time: {time.perf_counter()-t0:.1f}s')
print(f'\n✓ Pattern preprocessing complete — LEAKAGE FREE, MEMORY SAFE.')


Pass 2: Transforming and saving...
  10/30
  20/30
  30/30

Saved 30 separate files → data/pattern_features/batch_XXXX.parquet
Saved scalers + pattern_meta.pkl
Time: 956.7s

✓ Pattern preprocessing complete — LEAKAGE FREE, MEMORY SAFE.
